# Part 7 (실습) — 도구(Tool) 직접 정의하기

> **이 노트북의 목표**
> `@tool`로 리리마켓 도구를 직접 만들어 에이전트에 붙임. **docstring을 좋게/나쁘게 바꿔가며** 모델의 도구 선택이 달라지는 것을 관찰하고, `args_schema`로 입력을 통제함.
>
> **사용 모델**: `gemini-2.5-flash` · **선행**: Part 0~6

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
# @tool 데코레이터: 일반 파이썬 함수를 LangChain "도구"로 변환
#   → 함수의 docstring이 도구 설명이 되고, 파라미터가 도구 입력 스키마가 됨
#   → 에이전트가 이 설명을 읽고 언제 이 도구를 쓸지 스스로 판단함
from langchain_core.tools import tool
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-2.5-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print("준비 완료")

## 1. @tool — 함수를 도구로

함수 위에 `@tool` 한 줄이면 도구가 됨. 이름·docstring·타입 힌트가 자동으로 명세가 됨.

In [ ]:
@tool
def get_stock(product: str) -> str:
    """상품명을 받아 현재 재고 수량을 돌려준다."""
    stock_db = {"베이지 니트": 12, "트렌치 코트": 0, "린넨 셔츠": 5}
    qty = stock_db.get(product, "정보 없음")
    return f"{product}의 재고: {qty}"

# 도구가 모델에게 어떻게 보이는지 — 추출된 명세 확인
print("이름  :", get_stock.name)
print("설명  :", get_stock.description)
print("입력  :", get_stock.args)

### 에이전트에 붙여 호출
모델이 "재고를 알아야 하니 get_stock을 쓰자"고 스스로 판단함.

In [ ]:
agent = create_agent(model, tools=[get_stock])
# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
r = agent.invoke({"messages": [("user", "트렌치 코트 재고 있어?")]})
print(r["messages"][-1].content)

## 2. 핵심 실습 — docstring이 도구 선택을 좌우한다

모델은 **docstring만 읽고** 도구를 고름. 같은 기능도 설명이 모호하면 모델이 못 고름.

In [ ]:
# ❌ 나쁜 docstring — 무슨 일인지, 언제 쓰는지 불명확
@tool
def tool_bad(x: str) -> str:
    """처리한다."""
    rates = {"USD": 1380, "EUR": 1490}
    return f"{rates.get(x, '?')}원"

agent_bad = create_agent(model, tools=[tool_bad])
r = agent_bad.invoke({"messages": [("user", "1달러는 몇 원이야?")]})
print("[나쁜 docstring]", r["messages"][-1].content)
# → 모델이 tool_bad를 못 쓰거나 엉뚱하게 답할 수 있음

In [ ]:
# ✅ 좋은 docstring — 무엇을, 언제 쓰는지 명확
@tool
def get_exchange_rate(currency: str) -> str:
    """원화 대비 환율을 조회한다. currency는 'USD','EUR' 같은 통화 코드.
    해외 상품 가격을 원화로 환산하거나 환율을 물을 때 사용한다."""
    rates = {"USD": 1380, "EUR": 1490}
    return f"1 {currency} = {rates.get(currency, '정보 없음')}원"

agent_good = create_agent(model, tools=[get_exchange_rate])
r = agent_good.invoke({"messages": [("user", "1달러는 몇 원이야?")]})
print("[좋은 docstring]", r["messages"][-1].content)
# → 모델이 정확히 도구를 호출함

> 📌 **핵심**: 두 도구의 내부 코드는 거의 같음. 차이는 오직 docstring(과 이름·타입). 모델은 라벨(설명)만 읽고 공구를 고른다 — docstring이 곧 설계임.

## 3. 여러 도구 중 올바른 것 고르기

도구가 여럿이면, 모델은 docstring을 보고 상황에 맞는 것을 고름.

In [ ]:
@tool
def calculator(expression: str) -> str:
    """사칙연산 수식을 계산한다. 예: '12000 * 3'. 숫자 계산이 필요할 때 사용한다."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"계산 오류: {e}"

tools = [get_stock, get_exchange_rate, calculator]
multi_agent = create_agent(model, tools=tools)

# 질문마다 다른 도구를 골라야 함
for q in ["베이지 니트 재고 알려줘", "50달러는 원화로 얼마야?", "12000원짜리 3개 합계는?"]:
    r = multi_agent.invoke({"messages": [("user", q)]})
    # 어떤 도구를 썼는지 추적
    used = [tc["name"] for m in r["messages"] if getattr(m,"tool_calls",None) for tc in m.tool_calls]
    print(f"Q: {q}\n  → 쓴 도구: {used or '없음'}\n  → 답: {r['messages'][-1].content}\n")

## 4. args_schema — 입력 구조를 정밀하게

인수가 많거나 검증이 필요하면 Pydantic으로 입력 스키마를 정의함. 인수 설명·제약이 모델에 전달됨.

In [ ]:
# pydantic: 데이터 검증 라이브러리 — 입출력 데이터의 형식을 엄격하게 정의하고 검증
#   → BaseModel: 데이터 구조(스키마)를 정의하는 기본 클래스
#   → Field: 각 필드의 설명·기본값 등 메타데이터를 추가할 때 사용
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    category: str = Field(description="상품 카테고리 (예: 니트, 코트)")
    max_price: int = Field(description="최대 가격(원)", gt=0)

@tool(args_schema=SearchInput)
def search_products(category: str, max_price: int) -> str:
    """조건에 맞는 상품을 검색한다. 카테고리와 최대 가격으로 거를 때 사용한다."""
    catalog = [("베이지 니트", 49000), ("캐시미어 니트", 120000), ("울 코트", 89000)]
    hits = [n for n,p in catalog if category in n and p <= max_price]
    return f"검색 결과: {hits or '없음'}"

print("입력 스키마:", search_products.args)

agent_s = create_agent(model, tools=[search_products])
r = agent_s.invoke({"messages": [("user", "6만원 이하 니트 찾아줘")]})
print(r["messages"][-1].content)

> 📌 모델이 `category="니트"`, `max_price=60000`처럼 스키마에 맞춰 인수를 채움. `gt=0` 같은 제약도 명세로 전달됨.

## 5. 나쁜 설계 관찰 — 만능 도구의 함정

한 도구에 여러 기능을 욱여넣으면 모델이 혼란스러워함. (단일 기능 원칙)

In [ ]:
@tool
def do_everything(mode: str, value: str) -> str:
    """재고 조회, 환율 변환, 계산을 모두 한다. mode로 기능을 고른다."""
    # 모델이 mode를 무엇으로 줄지 헷갈리기 쉬움 — 설계가 모호
    return f"(mode={mode}, value={value} 처리)"

# 차라리 get_stock / get_exchange_rate / calculator 셋으로 나누는 게 모델이 고르기 쉬움
print("교훈: 도구는 '한 가지 일'만 — 만능 도구는 모델을 혼란시킴.")

## ✏️ 미니 실습

**과제**: 배송 상태를 조회하는 도구 `track_delivery`를 만들기.
- 인수: `order_id`(str)
- docstring에 "주문번호로 배송 상태를 조회한다. 배송이 어디쯤인지 물을 때 사용한다." 명시
- 에이전트에 붙여 "주문 A123 배송 어디쯤이야?" 호출

In [ ]:
# TODO: 직접 작성
# @tool
# def track_delivery(order_id: str) -> str:
#     """..."""
#     status = {"A123": "배송 중 (수도권)"}
#     return ...
# agent_t = create_agent(model, tools=[track_delivery])
# print(agent_t.invoke({"messages":[("user","주문 A123 배송 어디쯤이야?")]})["messages"][-1].content)

<details><summary>👉 정답</summary>

```python
@tool
def track_delivery(order_id: str) -> str:
    """주문번호로 배송 상태를 조회한다. 배송이 어디쯤인지 물을 때 사용한다."""
    status = {"A123": "배송 중 (수도권)", "A124": "배송 완료"}
    return f"주문 {order_id}: {status.get(order_id, '정보 없음')}"

agent_t = create_agent(model, tools=[track_delivery])
print(agent_t.invoke({"messages":[("user","주문 A123 배송 어디쯤이야?")]})["messages"][-1].content)
```
</details>

## 정리

| 절 | 한 일 | 핵심 |
|---|---|---|
| 1 | `@tool` 기본 | 이름·docstring·타입 자동 명세화 |
| 2 | docstring 품질 비교 | 설명이 도구 선택을 좌우 |
| 3 | 여러 도구 선택 | 모델이 상황에 맞게 고름 |
| 4 | `args_schema` | Pydantic으로 입력 통제 |
| 5 | 나쁜 설계 | 만능 도구의 함정 |

### 3줄 요약
1. `@tool`로 함수를 도구화하면 이름·docstring·타입이 모델용 명세가 됨.
2. 모델은 docstring만 읽고 고르므로, 설명이 곧 설계임.
3. 단일 기능·명확한 설명·정밀한 입력 스키마가 좋은 도구의 조건임.

### ▶ 다음 (Part 8)
가장 강력한 도구 — "우리 문서에서 찾아 답하는" **RAG**의 기초. 문서 로더 → 청킹 → 임베딩 → 벡터스토어 → retriever.